In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

In [3]:
df = pd.read_csv(
    "D:/SALEM_AI/Machine Learning Models/3-Priority & Resource Allocation Model/Dataset/priority_allocation_dataset_v2.csv"
                 )
print("Dataset Shape:", df.shape)

Dataset Shape: (12000, 14)


In [4]:
# Encode Categorical Columns
label_encoders = {}

categorical_cols = [
    "severity_label",
    "assigned_team",
    "suggested_resources",
    "priority_level"
]

for col in categorical_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    label_encoders[col] = le

In [5]:
# Features & Target
X = df[[
    "severity_label",
    "severity_score",
    "area_load",
    "available_teams",
    "team_skill_match",
    "historical_team_performance",
    "citizen_trust_score",
    "reports_nearby_1h"
]]

y = df["priority_level"]

In [6]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)



Train Shape: (9600, 8)
Test Shape: (2400, 8)


In [7]:
# Model Training
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

print("\n✅ Model Training Completed")



✅ Model Training Completed


In [9]:
# Predictions
y_pred = model.predict(X_test)

In [10]:
# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print("\n=================================================")
print("MODEL EVALUATION")
print("=================================================")

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:\n")

target_names = label_encoders[
    "priority_level"
].classes_

print(classification_report(
    y_test,
    y_pred,
    target_names=target_names
))


MODEL EVALUATION

Accuracy: 0.8404

Classification Report:

              precision    recall  f1-score   support

          P1       0.81      0.82      0.81       423
          P2       0.83      0.82      0.83      1109
          P3       0.87      0.87      0.87       868

    accuracy                           0.84      2400
   macro avg       0.84      0.84      0.84      2400
weighted avg       0.84      0.84      0.84      2400



In [12]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:\n")
print(cm)


Confusion Matrix:

[[346  77   0]
 [ 81 912 116]
 [  0 109 759]]


In [13]:
# Feature Importance
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\n=================================================")
print("FEATURE IMPORTANCE")
print("=================================================\n")

print(importance_df)



FEATURE IMPORTANCE

                       Feature  Importance
1               severity_score    0.411945
0               severity_label    0.304157
2                    area_load    0.128612
5  historical_team_performance    0.047683
6          citizen_trust_score    0.047533
4             team_skill_match    0.031169
7            reports_nearby_1h    0.019075
3              available_teams    0.009826


In [14]:
# Save Model

joblib.dump(
    model,
    "D:/SALEM_AI/Machine Learning Models/3-Priority & Resource Allocation Model/modelpk/priority_allocation_model.pkl"
)

joblib.dump(
    label_encoders,
    "D:/SALEM_AI/Machine Learning Models/3-Priority & Resource Allocation Model/modelpk/priority_label_encoders.pkl"
)

print("\n✅ Model Saved Successfully")


✅ Model Saved Successfully


In [ ]:
joblib.dump(
    X.columns.tolist(),
    "D:/SALEM_AI/Machine Learning Models/3-Priority & Resource Allocation Model/modelpk/priority_features.pkl"
)

['priority_features.pkl']

In [15]:
# Sample Prediction
sample = X_test.iloc[[0]]

prediction = model.predict(sample)[0]

decoded_prediction = label_encoders[
    "priority_level"
].inverse_transform([prediction])[0]

print("\n=================================================")
print("SAMPLE PREDICTION")
print("=================================================\n")

print("Predicted Priority:", decoded_prediction)


SAMPLE PREDICTION

Predicted Priority: P2
